# Parte 3: motor de recomendações

> **Pergunta do case:** usando os insights das Partes 1 e 2, construa um sistema simples de recomendações que analisa uma campanha e gera sugestões de melhoria. Requisitos: (1) recebe uma linha do dataset e gera pelo menos 3 recomendações; (2) acionáveis e quantificadas; (3) contextuais (plataforma, categoria e outros atributos); (4) ordenadas por impacto estimado; (5) protótipo funcional.

**A ideia:** o Modelo A da Parte 2 prevê a nota de um criativo **antes** de postar. Então dá para perguntar a ele "e se este mesmo vídeo tivesse um gancho?": mudo só aquela coluna, peço a previsão de novo, e a diferença é o ganho estimado. É como um simulador de "e se". A Parte 1 completa com o efeito observado na métrica que importa para a campanha (CTR, engajamento ou retenção), **na rede dela**.

| Passo | O que mostra |
|---|---|
| 1 | De onde vem cada número de uma recomendação (por extenso, para uma campanha) |
| 2 | Quais mudanças o motor considera, e por que essas |
| 3 | O motor completo, conferido contra o passo a passo |
| 4 | O motor em campanhas que ele nunca viu (uma de cada rede) |
| 5 | É contextual? O mesmo vídeo em três redes |
| 6 | O motor se comporta como esperado? Conferências em 100 campanhas |

O motor fica em `src/recomendacoes.py` e também roda pela linha de comando: `python -m src.recomendacoes KLK-0479`.

## Passo 0: preparação

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path("../..")
sys.path.append(str(RAIZ))
from src.features import COLUNAS_A
from src.limpeza import carregar, separar_treino_teste
from src.recomendacoes import METRICA_DO_OBJETIVO, carregar_motor, mudancas_possiveis, preparar, recomendar

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 230)

# O motor usa só o que foi aprendido no treino: o Modelo A (Parte 2) e a tabela de efeitos por rede (Parte 1)
motor = carregar_motor()
_, teste_bruto = separar_treino_teste(carregar(RAIZ / "data/raw/klike_challenge_dataset.csv"))
print(f"Modelo: {type(motor['modelo'][-1]).__name__} | efeitos da Parte 1: {len(motor['efeitos'])} linhas")
print("Formatos que cada rede usa com pelo menos 15 criativos:", motor["formatos_da_rede"])

Modelo: LinearRegression | efeitos da Parte 1: 72 linhas
Formatos que cada rede usa com pelo menos 15 criativos: {'LinkedIn': ['horizontal', 'quadrado'], 'Meta': ['horizontal', 'quadrado', 'vertical'], 'TikTok': ['quadrado', 'vertical']}


## Passo 1: de onde vem cada número de uma recomendação

Uma campanha do **teste** (criativos que o modelo nunca viu), tal como está no dataset original:

In [2]:
linha = teste_bruto[(teste_bruto["platform"] == "TikTok") & ~teste_bruto["has_hook"].astype(bool)].iloc[0]
campanha = preparar(linha, motor)  # a mesma limpeza das Partes 1 e 2
campanha[["campaign_id", "platform", "category", "objective", "has_hook", "has_face", "has_cta", "has_subtitle",
          "text_density", "format", "music_voice_ratio", "is_retargeting", "klike_score"]].to_frame("valor").T

,campaign_id,platform,category,objective,has_hook,has_face,has_cta,has_subtitle,text_density,format,music_voice_ratio,is_retargeting,klike_score
valor,KLK-0419,TikTok,Branding,awareness,0,1,1,sim,high,vertical,0.29,1,55.8


**(a) O ganho na nota, pelo modelo.** Prevejo a nota do criativo como ele é; depois copio a linha, ligo o gancho e prevejo de novo. Como tudo o resto fica igual, a diferença é o efeito do gancho **para este criativo** (no Modelo A, a música × TikTok faz o efeito depender da rede).

In [3]:
def prever(c):
    return float(np.clip(motor["modelo"].predict(pd.DataFrame([c])[COLUNAS_A])[0], 0, 100))


com_gancho = campanha.copy()
com_gancho["has_hook"] = 1
nota_hoje, nota_com_gancho = prever(campanha), prever(com_gancho)
print(f"Nota real: {campanha['klike_score']:.1f} | prevista hoje: {nota_hoje:.1f} | prevista com gancho: {nota_com_gancho:.1f} "
      f"| ganho: {nota_com_gancho - nota_hoje:+.1f} pontos")

Nota real: 55.8 | prevista hoje: 57.6 | prevista com gancho: 75.5 | ganho: +17.9 pontos


**(b) O efeito na métrica da campanha, pela Parte 1.** Cada objetivo quer mover uma métrica (Parte 1, pergunta 1.2): awareness → retenção, engagement → engajamento, traffic → CTR. Em conversions e app install, a Parte 1 mostrou que o criativo não mexe no custo por conversão; o efeito mostrado é no CTR, a etapa anterior. Procuro, na tabela `efeitos_por_plataforma.csv` (Parte 1, pergunta 1.3), a diferença das medianas com e sem gancho **nessa rede e nessa métrica**:

In [4]:
metrica = METRICA_DO_OBJETIVO[campanha["objective"]]
efeitos = motor["efeitos"]
linha_efeito = efeitos[(efeitos["plataforma"] == campanha["platform"]) & (efeitos["atributo"] == "Gancho (hook)")
                       & (efeitos["metrica"] == metrica)]
linha_efeito

,plataforma,atributo,metrica,mediana_com,mediana_sem,diferenca_pct,n_com,n_sem,intervalo_baixo_pct,intervalo_alto_pct,confiavel
26,TikTok,Gancho (hook),Retenção,0.559341,0.423077,32.207792,56,63,16.9656,73.940217,True


A recomendação junta os dois: "vídeos parecidos com gancho têm nota ~X pontos maior; no TikTok, vídeos com gancho têm CTR entre +A% e +B% maior (comparação simples, N vídeos)". Os dois números têm precisões diferentes, e o texto diz isso. A nota vem da regressão, que compara vídeos parecidos. A métrica vem de uma comparação simples de medianas feita em poucos vídeos, então aparece como **faixa** (o intervalo de 95% do bootstrap), arredondada de 5 em 5, e com o número de vídeos. Ela só aparece se a faixa inteira ficar de um lado do zero (coluna `confiavel`) e se os dois grupos tiverem pelo menos 15 criativos. Um efeito que não passa nesse filtro pode até ter o sinal oposto ao da nota (por exemplo, o CTA anda junto com nota maior, mas no engajamento do TikTok dava −11%), e confundiria quem lê.

## Passo 2: quais mudanças o motor considera?

Só o que o time controla e o que as Partes 1 e 2 sustentam:

| Mudança | Quando aparece | Base |
|---|---|---|
| adicionar gancho, rosto, CTA, legenda | o criativo não tem | os maiores efeitos da Parte 2 (+8 a +18 pontos) |
| texto na tela para "médio" | está em "pouco" ou "muito" | ponto ótimo no meio (Partes 1 e 2) |
| trocar música por voz | música/voz acima de 0,3 no TikTok, ou de 0,5 nas outras redes | só pesa no TikTok (Parte 2) |
| mudar o formato | a rede usa o outro formato com pelo menos 15 criativos | efeito pequeno; o TikTok não tem horizontal e o LinkedIn tem só 9 verticais |

**Fica de fora:**
- **a duração:** a Parte 1 não achou duração ideal, e na Parte 2 ela vale 0,02 ponto por segundo. Recomendar "reduza de 45 para 20 s" não teria base nestes dados;
- **trocar de plataforma, categoria ou objetivo:** não é melhorar a campanha, é fazer outra;
- **a idade do público:** a Parte 1 não achou padrão claro;
- **mudanças que somam menos de 1 ponto de nota:** pedir ao time um trabalho que não muda nada não é uma recomendação útil.

**O retargeting fica à parte.** Campanhas que miram quem já conhece a marca têm nota cerca de 4 pontos maior (Parte 2), mas isso é uma decisão de público, não uma mudança no vídeo. Por isso ele não disputa lugar com as recomendações do vídeo: aparece no fim, numa linha própria, como "sugestão de público".

**Prioridade:** pela nota prevista, que é o alvo da Parte 2 e resume o funil.

**E se o vídeo já fizer quase tudo certo?** O motor não inventa melhoria sem base. Se sobrarem menos de 3, ele completa com o que o vídeo **deve manter**, também quantificado ("manter o gancho: vídeos parecidos sem ele têm nota ~18 pontos menor").

As mudanças possíveis para a campanha do passo 1:

In [5]:
pd.DataFrame(mudancas_possiveis(campanha, motor), columns=["o que fazer", "colunas alteradas", "atributo na Parte 1", "direção",
                                                         "grupo na frase da métrica"])

,o que fazer,colunas alteradas,atributo na Parte 1,direção,grupo na frase da métrica
0,Adicionar um gancho nos 3 primeiros segundos,{'has_hook': 1},Gancho (hook),adicionar,vídeos com gancho
1,Mudar o texto na tela de 'muito' para 'médio',{'text_density': 'medium'},Muito texto,tirar,vídeos sem excesso de texto
2,Mudar o formato de vertical para quadrado,{'format': 'quadrado'},NaN,NaN,NaN


## Passo 3: o motor completo

`recomendar` faz os passos 1 e 2 para todas as mudanças possíveis, descarta as que somam menos de 1 ponto de nota, ordena pelo ganho e devolve as 3 primeiras para o vídeo. Se a campanha não usa retargeting, acrescenta no fim, à parte, a sugestão de público. Confiro que o ganho do gancho é o mesmo calculado à mão no passo 1:

In [6]:
recomendacoes = recomendar(linha, motor)
gancho = recomendacoes[recomendacoes["recomendação"].str.startswith("Adicionar um gancho")]
assert np.isclose(gancho["impacto na nota"].iloc[0], nota_com_gancho - nota_hoje)
recomendacoes

,tipo,recomendação,impacto na nota
prioridade,,,
1,melhorar,"Adicionar um gancho nos 3 primeiros segundos: vídeos parecidos com essa mudança têm nota ~18 pontos maior (de 58 para ~76); no TikTok, vídeos com gancho têm retenção entre +15% e +75% maior (comparação simples, 119 vídeos).",17.922319
2,melhorar,"Mudar o texto na tela de 'muito' para 'médio': vídeos parecidos com essa mudança têm nota ~15 pontos maior (de 58 para ~73); no TikTok, vídeos sem excesso de texto têm retenção entre +5% e +50% maior (comparação simples, 119 v...",15.089107
3,manter,Manter o rosto humano: vídeos parecidos sem isso têm nota ~12 pontos menor (de 58 para ~46).,11.528615


## Passo 4: o motor em campanhas que ele nunca viu

Uma campanha do teste de cada rede, sorteada entre as que têm nota real abaixo da mediana (as que mais precisam de ajuda):

In [7]:
mediana = teste_bruto["klike_score"].median()
for rede in ["Meta", "TikTok", "LinkedIn"]:
    exemplo = teste_bruto[(teste_bruto["platform"] == rede) & (teste_bruto["klike_score"] < mediana)].sample(1, random_state=7).iloc[0]
    print(f"\n{exemplo['campaign_id']} | {rede} | {exemplo['category']} | objetivo: {exemplo['objective']} | nota real: {exemplo['klike_score']:.1f}")
    for prioridade, r in recomendar(exemplo, motor).iterrows():
        print(f"  {prioridade}. [{r['tipo']}] {r['recomendação']}")


KLK-0339 | Meta | App Install | objetivo: app_install | nota real: 44.0


  1. [melhorar] Mostrar um rosto humano no vídeo: vídeos parecidos com essa mudança têm nota ~12 pontos maior (de 52 para ~63).
  2. [melhorar] Colocar legenda: vídeos parecidos com essa mudança têm nota ~8 pontos maior (de 52 para ~60).
  3. [melhorar] Mudar o texto na tela de 'pouco' para 'médio': vídeos parecidos com essa mudança têm nota ~8 pontos maior (de 52 para ~59).

KLK-0268 | TikTok | Lead Gen | objetivo: engagement | nota real: 47.0


  1. [melhorar] Adicionar um gancho nos 3 primeiros segundos: vídeos parecidos com essa mudança têm nota ~18 pontos maior (de 48 para ~66).


  2. [melhorar] Mudar o texto na tela de 'muito' para 'médio': vídeos parecidos com essa mudança têm nota ~15 pontos maior (de 48 para ~63).
  3. [melhorar] Terminar com uma chamada para ação (CTA): vídeos parecidos com essa mudança têm nota ~8 pontos maior (de 48 para ~56).
  4. [público] Sugestão de público (não é mudança no vídeo): campanhas que também miram quem já conhece a marca (retargeting) têm nota ~4 pontos maior.

KLK-0479 | LinkedIn | SaaS | objetivo: traffic | nota real: 53.0
  1. [melhorar] Adicionar um gancho nos 3 primeiros segundos: vídeos parecidos com essa mudança têm nota ~18 pontos maior (de 49 para ~67); no LinkedIn, vídeos com gancho têm CTR entre +25% e +355% maior (comparação simples, 66 vídeos).
  2. [melhorar] Mostrar um rosto humano no vídeo: vídeos parecidos com essa mudança têm nota ~12 pontos maior (de 49 para ~61).
  3. [manter] Manter o texto na tela em nível médio: vídeos parecidos sem isso têm nota ~15 pontos menor (de 49 para ~34).
  4. [público] Su

**Lendo os exemplos:**

- **Cada campanha recebe 3 recomendações para o vídeo, conforme o que falta nele:** no Meta, rosto, legenda e texto; no TikTok, gancho, texto e CTA; no LinkedIn, gancho, rosto e "manter o texto em nível médio" (o vídeo já acerta nisso). Quando a campanha não usa retargeting, vem no fim, à parte, a sugestão de público.
- **A ordem segue o tamanho do efeito na nota:** gancho (~18 pontos), muito texto para médio (~15), rosto (~12), legenda e CTA (~8), pouco texto para médio (~8).
- **O efeito na métrica só aparece quando a Parte 1 tem um padrão confiável, e vem como faixa.** Na campanha de traffic do LinkedIn, o gancho vem com "CTR entre +25% e +355% maior (comparação simples, 66 vídeos)": a direção é clara, mas o tamanho é muito incerto, e o texto diz isso. Na de engagement do TikTok, nenhum efeito no engajamento passa no filtro, então só a nota aparece.

## Passo 5: é contextual? O mesmo vídeo em três redes

Pego um criativo do teste com muita música e pouca voz, sem gancho, e simulo que ele roda em cada rede (mudo só a plataforma; o formato vai para o mais comum da rede, porque o TikTok não tem horizontal):

In [8]:
base = teste_bruto[(teste_bruto["music_voice_ratio"] > 0.7) & ~teste_bruto["has_hook"].astype(bool)].iloc[0].copy()
FORMATO_MAIS_COMUM = {"Meta": "vertical", "TikTok": "vertical", "LinkedIn": "horizontal"}
for rede in ["Meta", "TikTok", "LinkedIn"]:
    versao = base.copy()
    versao["platform"], versao["format"] = rede, FORMATO_MAIS_COMUM[rede]
    print(f"\n{rede} (música/voz = {versao['music_voice_ratio']:.2f}, objetivo: {versao['objective']}):")
    for prioridade, r in recomendar(versao, motor).iterrows():
        print(f"  {prioridade}. [{r['tipo']}] {r['recomendação']}")


Meta (música/voz = 0.92, objetivo: conversions):
  1. [melhorar] Adicionar um gancho nos 3 primeiros segundos: vídeos parecidos com essa mudança têm nota ~18 pontos maior (de 55 para ~73); no Meta, vídeos com gancho têm CTR entre +5% e +85% maior (comparação simples, 193 vídeos).
  2. [melhorar] Mudar o texto na tela de 'muito' para 'médio': vídeos parecidos com essa mudança têm nota ~15 pontos maior (de 55 para ~70); no Meta, vídeos sem excesso de texto têm CTR entre +5% e +210% maior (comparação simples, 193 vídeos).
  3. [manter] Manter o rosto humano: vídeos parecidos sem isso têm nota ~12 pontos menor (de 55 para ~43).
  4. [público] Sugestão de público (não é mudança no vídeo): campanhas que também miram quem já conhece a marca (retargeting) têm nota ~4 pontos maior.

TikTok (música/voz = 0.92, objetivo: conversions):


  1. [melhorar] Adicionar um gancho nos 3 primeiros segundos: vídeos parecidos com essa mudança têm nota ~18 pontos maior (de 51 para ~69); no TikTok, vídeos com gancho têm CTR entre +20% e +165% maior (comparação simples, 141 vídeos).
  2. [melhorar] Mudar o texto na tela de 'muito' para 'médio': vídeos parecidos com essa mudança têm nota ~15 pontos maior (de 51 para ~66).
  3. [melhorar] Trocar parte da música por voz (música/voz de 0.9 para 0.3): vídeos parecidos com essa mudança têm nota ~7 pontos maior (de 51 para ~58).
  4. [público] Sugestão de público (não é mudança no vídeo): campanhas que também miram quem já conhece a marca (retargeting) têm nota ~4 pontos maior.

LinkedIn (música/voz = 0.92, objetivo: conversions):


  1. [melhorar] Adicionar um gancho nos 3 primeiros segundos: vídeos parecidos com essa mudança têm nota ~18 pontos maior (de 49 para ~67); no LinkedIn, vídeos com gancho têm CTR entre +25% e +355% maior (comparação simples, 66 vídeos).
  2. [melhorar] Mudar o texto na tela de 'muito' para 'médio': vídeos parecidos com essa mudança têm nota ~15 pontos maior (de 49 para ~64).
  3. [melhorar] Mudar o formato de horizontal para quadrado: vídeos parecidos com essa mudança têm nota ~1 ponto maior (de 49 para ~50).
  4. [público] Sugestão de público (não é mudança no vídeo): campanhas que também miram quem já conhece a marca (retargeting) têm nota ~4 pontos maior.


**O que muda com o contexto, e o que não muda:**

- **O ganho na nota é quase o mesmo em qualquer rede:** o gancho soma ~18 pontos no Meta, no TikTok e no LinkedIn, porque o modelo da Parte 2 soma os efeitos e só tem uma combinação com a rede (a música no TikTok). Esse é o limite do contexto neste motor, e é coerente com a Parte 1, que mostrou que o que funciona é quase igual nas três redes.
- **O contexto entra de três jeitos:**
  - **a voz no TikTok:** só lá aparece "trocar música por voz" (~7 pontos); nas outras redes, a música não pesa;
  - **a métrica mostrada, que depende da rede e do objetivo:** o CTR dos vídeos com gancho é entre +5% e +85% maior no Meta, entre +20% e +165% no TikTok e entre +25% e +355% no LinkedIn. A faixa de "sem excesso de texto" só aparece no Meta, a única rede em que ela é confiável;
  - **as mudanças possíveis:** o formato sugerido é sempre um que a rede usa (no LinkedIn, quadrado; nunca vertical, que lá quase não existe).
- **A categoria não entra:** ela saiu do modelo porque repete o objetivo (notebook 2.1). O objetivo, por sua vez, escolhe a métrica mostrada (aqui, conversions → CTR).

## Passo 6: o motor se comporta como esperado?

Não dá para provar que as recomendações **funcionam** sem testar: para isso seria preciso um teste A/B (o mesmo anúncio com e sem a mudança, para públicos sorteados). O que dá para conferir, nas 100 campanhas do teste, é se o motor se comporta como deveria:

- **(a) Cobertura:** toda campanha recebe 3 recomendações para o vídeo? Quantas são melhorias e quantas são "manter"?
- **(b) Quais aparecem em primeiro?** Devem ser os maiores efeitos (gancho, rosto, texto).
- **(c) O "espaço para melhorar" acompanha a nota?** Somo os ganhos das melhorias de cada campanha. Atenção: esse ganho é calculado pelo próprio modelo, então ele acompanha a nota **prevista** quase por construção. Se ele também acompanha a nota **real**, isso só repete o quanto o modelo acerta (notebook 2.3). Mostra que o motor é consistente com o modelo, mas **não** valida as recomendações.

In [9]:
linhas = []
for _, campanha_teste in teste_bruto.iterrows():
    r = recomendar(campanha_teste, motor)
    do_video = r[r["tipo"] != "público"]
    melhorias = r[r["tipo"] == "melhorar"]
    linhas.append({"campaign_id": campanha_teste["campaign_id"], "rede": campanha_teste["platform"], "nota real": campanha_teste["klike_score"],
                   "nota prevista": prever(preparar(campanha_teste, motor)),
                   "recomendações do vídeo": len(do_video), "melhorias": len(melhorias), "sugestão de público": (r["tipo"] == "público").any(),
                   "ganho potencial": melhorias["impacto na nota"].sum(), "primeira": r["recomendação"].iloc[0].split(":")[0]})
no_teste = pd.DataFrame(linhas)
print("Recomendações do vídeo por campanha:", no_teste["recomendações do vídeo"].value_counts().to_dict())
print("Melhorias por campanha:", no_teste["melhorias"].value_counts().sort_index().to_dict())
print(f"Campanhas com sugestão de público (retargeting): {no_teste['sugestão de público'].sum()}")
com_prevista = no_teste["ganho potencial"].corr(no_teste["nota prevista"], method="spearman")
com_real = no_teste["ganho potencial"].corr(no_teste["nota real"], method="spearman")
prevista_real = no_teste["nota prevista"].corr(no_teste["nota real"], method="spearman")
print(f"Spearman do ganho potencial com a nota prevista: {com_prevista:.2f} | com a nota real: {com_real:.2f} "
      f"| nota prevista × real: {prevista_real:.2f}")
no_teste["primeira"].value_counts().rename("vezes em 1º lugar").to_frame()

Recomendações do vídeo por campanha: {3: 100}
Melhorias por campanha: {0: 5, 1: 12, 2: 20, 3: 63}
Campanhas com sugestão de público (retargeting): 80
Spearman do ganho potencial com a nota prevista: -0.95 | com a nota real: -0.77 | nota prevista × real: 0.81


,vezes em 1º lugar
primeira,
Adicionar um gancho nos 3 primeiros segundos,50
Mostrar um rosto humano no vídeo,14
Mudar o texto na tela de 'muito' para 'médio',11
Colocar legenda,10
Manter o gancho nos 3 primeiros segundos,5
Mudar o texto na tela de 'pouco' para 'médio',5
Terminar com uma chamada para ação (CTA),3
Mudar o formato de horizontal para vertical,1
Mudar o formato de quadrado para vertical,1


In [10]:
# Nota real por faixa de ganho potencial (4 faixas com o mesmo número de campanhas)
no_teste["faixa de ganho"] = pd.qcut(no_teste["ganho potencial"].rank(method="first"), 4,
                                     labels=["menor espaço", "2º quarto", "3º quarto", "maior espaço"])
no_teste.groupby("faixa de ganho", observed=True).agg(
    campanhas=("nota real", "size"), ganho_potencial_medio=("ganho potencial", "mean"), nota_real_mediana=("nota real", "median")).round(1)

,campanhas,ganho_potencial_medio,nota_real_mediana
faixa de ganho,,,
menor espaço,25,7.6,80.3
2º quarto,25,23.0,61.4
3º quarto,25,30.8,53.8
maior espaço,25,38.5,48.0


**Lendo as conferências:**

- **(a) As 100 campanhas recebem 3 recomendações para o vídeo:** 63 com 3 melhorias, 20 com 2, 12 com 1 e 5 com nenhuma. As que têm menos de 3 já seguem quase tudo e são completadas com "manter". 80 delas recebem também a sugestão de público.
- **(b) Em 1º lugar aparecem os maiores efeitos:** gancho (50 vezes), rosto (14), muito texto (11), legenda (10). Em 5 campanhas, o 1º lugar é "manter o gancho": o vídeo já acerta no que mais importa.
- **(c) O espaço para melhorar acompanha a nota, mas isso não valida o motor.** O ganho potencial anda junto com a nota prevista (Spearman de −0,95) quase por construção, porque os dois vêm do mesmo modelo. Ele também anda junto com a nota real (−0,77), mas só porque o modelo acerta a nota (0,81 entre prevista e real). A nota real mediana é 80 no quarto com menos espaço para melhorar e 48 no quarto com mais espaço.
- **Conclusão honesta:** o motor é consistente com o modelo e se comporta como esperado. Se seguir as recomendações realmente melhora uma campanha, só um teste A/B pode dizer.

## Resposta à pergunta da Parte 3

> 💡 **Conclusão**
>
> - **1. Input e 3 recomendações:** recebe uma linha do dataset original (`recomendar(linha, motor)`, ou `python -m src.recomendacoes KLK-0479`) e devolve sempre 3 recomendações para o vídeo (100 de 100 campanhas do teste), mais uma sugestão de público quando a campanha não usa retargeting.
> - **2. Acionáveis e quantificadas:** cada uma diz o que mudar e quantos pontos de nota vídeos parecidos com a mudança têm a mais (de X para ~Y). Quando a Parte 1 tem padrão confiável, mostra também a faixa do efeito na métrica do objetivo, com o número de vídeos (por exemplo, gancho no LinkedIn: ~18 pontos e CTR entre +25% e +355% maior, em 66 vídeos).
> - **3. Contextuais, com um limite:** a voz só é sugerida no TikTok, a métrica mostrada depende da rede e do objetivo, e o formato sugerido é sempre um que a rede usa. O ganho na nota, porém, é quase o mesmo em todas as redes, e a categoria não entra (repete o objetivo).
> - **4. Priorização:** em ordem de ganho na nota prevista; mudanças de menos de 1 ponto não entram; o que deve ser mantido vem depois; a sugestão de público fica por último, à parte.
> - **5. Protótipo funcional:** função Python, linha de comando e este notebook, com autoconferência (`python -m src.recomendacoes`).
>
> O motor é consistente com o modelo, mas isso não prova que as recomendações funcionam: é associação em dados simulados, e só um teste A/B validaria.